# CPU Deployment & Quantization Benchmark — Falconsai T5 Bullet Specialist

Model: `JayShah07/falconai-text-bullet-t5`

This Colab notebook benchmarks the same fine-tuned T5 specialist on **CPU only** and compares **PyTorch FP32**, **ONNX Runtime FP32**, and **ONNX Runtime dynamic INT8** on the same external evaluation CSV.

It measures model/artifact size, load time, mean/median/p95 latency, output tokens, tokens/sec, ROUGE-1/2/L, BERTScore P/R/F1, bullet format, bullet-count error, and compression. All candidates use the same prompt, deterministic generation, post-processing, and 500-row evaluation set.

The deployment sequence tested here is: `encoder pass -> decoder first step -> cached autoregressive decode`. `use_cache=True` enables the decoder KV-cache path where supported.

In [ ]:
# ============================================================
# CELL 1A — INSTALL SAME STACK AS WORKING GPU EVALUATION
# ============================================================

!pip install -q -U \
    transformers \
    accelerate \
    sentencepiece \
    pandas \
    tqdm \
    rouge-score \
    bert-score \
    psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [ ]:
# CELL 2 — Imports
import os, gc, time, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from bert_score import BERTScorer

print("PyTorch:", torch.__version__)

PyTorch: 2.11.0+cpu


In [ ]:
# CELL 3 — Force CPU + reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cpu"
print("CUDA available:", torch.cuda.is_available())
print("Benchmark device:", DEVICE)
print("Logical CPUs:", psutil.cpu_count(logical=True))
print("Physical CPUs:", psutil.cpu_count(logical=False))
print("System RAM GB:", round(psutil.virtual_memory().total / 1024**3, 2))

CUDA available: False
Benchmark device: cpu
Logical CPUs: 2
Physical CPUs: 1
System RAM GB: 12.67


In [ ]:
# CELL 4 — Configuration
MODEL_ID = "JayShah07/falconai-text-bullet-t5"
EVAL_CSV = "/content/output_with_bullet_points.csv"  # change only if your filename differs
MAX_INPUT_LENGTH = 2048
MAX_OUTPUT_LENGTH = 256
MAX_EVAL_ROWS = 500
THREAD_CANDIDATES = [1, 2, 4, 8]
BASE_DIR = Path("/content/t5_cpu_benchmark")
ONNX_FP32_DIR = BASE_DIR / "onnx_fp32"
ONNX_INT8_DIR = BASE_DIR / "onnx_int8"
RESULTS_DIR = BASE_DIR / "results"
for d in [BASE_DIR, ONNX_FP32_DIR, ONNX_INT8_DIR, RESULTS_DIR]: d.mkdir(parents=True, exist_ok=True)
print(MODEL_ID)

JayShah07/falconai-text-bullet-t5


In [ ]:
# CELL 5 — Exact task instruction used for SFT
TASK_INSTRUCTION = """
Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:
""".strip()
BULLET_TOKEN = "<BULLET>"
def build_encoder_text(text): return TASK_INSTRUCTION + "\n" + str(text).strip()

In [ ]:
# CELL 6 — Load and clean evaluation CSV
REQUIRED_COLUMNS = ["text", "source", "example_id", "bullet_points"]
def clean_dataframe(df):
    df=df.copy().dropna(subset=["text","bullet_points"])
    df["text"]=df["text"].astype(str).str.strip(); df["bullet_points"]=df["bullet_points"].astype(str).str.strip()
    return df[(df["text"]!="") & (df["bullet_points"]!="")].reset_index(drop=True)
if not os.path.exists(EVAL_CSV): raise FileNotFoundError(f"Upload the same evaluation CSV to {EVAL_CSV}")
eval_df=clean_dataframe(pd.read_csv(EVAL_CSV))
for c in REQUIRED_COLUMNS: assert c in eval_df.columns, f"Missing: {c}"
eval_df=eval_df.iloc[:MAX_EVAL_ROWS].copy().reset_index(drop=True)
print("Evaluation rows:", len(eval_df)); display(eval_df.head())

Evaluation rows: 500


,text,source,example_id,bullet_points
0,Powell: N. Korea Blast Not Nuclear Event The U...,ag_news,0,- Powell says the large North Korean explosion...
1,Jerusalem (CNN) -- Two attacks carried out aga...,cnn_dailymail,1,- Two recent attacks on Palestinians sparked I...
2,Former Kan. Junior College Coach Indicted (AP)...,ag_news,2,- Former Kansas junior college basketball coac...
3,The Ethiopian Airlines flight was travelling f...,xsum,3,- Ethiopian Airlines Flight ET500 was travelin...
4,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...",cnn_dailymail,4,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
# CELL 7 — Shared post-processing and metric helpers
def postprocess_generated_text(raw_text):
    text=str(raw_text).strip()
    if BULLET_TOKEN in text:
        pieces=[p.strip() for p in text.split(BULLET_TOKEN) if p.strip()]
        return "\n".join("- "+p for p in pieces)
    lines=[x.strip() for x in text.splitlines() if x.strip()]
    if lines and all(x.startswith("- ") for x in lines): return "\n".join(lines)
    return "- "+text if text else ""
def count_bullets(text): return sum(x.strip().startswith("- ") for x in str(text).splitlines())
def bullet_format_score(text):
    lines=[x.strip() for x in str(text).splitlines() if x.strip()]
    return 0.0 if not lines else sum(x.startswith("- ") for x in lines)/len(lines)
def word_count(text): return len(str(text).split())
def directory_size_mb(path):
    p=Path(path); return sum(x.stat().st_size for x in p.rglob("*") if x.is_file())/1024**2
def process_rss_mb(): return psutil.Process(os.getpid()).memory_info().rss/1024**2
rouge=rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)

In [ ]:
# CELL 8 — Load tokenizer once
start=time.perf_counter()
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer_load_seconds=time.perf_counter()-start
bullet_token_id=tokenizer.convert_tokens_to_ids(BULLET_TOKEN)
print("Tokenizer load s:", round(tokenizer_load_seconds,3))
print("BULLET ID:",bullet_token_id,"UNK:",tokenizer.unk_token_id)
assert bullet_token_id != tokenizer.unk_token_id, "<BULLET> missing from tokenizer"

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

Tokenizer load s: 5.329
BULLET ID: 32100 UNK: 2


## Candidate A — PyTorch FP32 CPU baseline

This establishes the reference quality and CPU latency. Quantized candidates must be compared against this exact output contract.

In [ ]:
# CELL 9 — Load PyTorch FP32 model on CPU
gc.collect(); rss0=process_rss_mb(); start=time.perf_counter()
pt_model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to("cpu"); pt_model.eval()
pt_load_seconds=time.perf_counter()-start; rss1=process_rss_mb()
print("Load s:",round(pt_load_seconds,3),"RSS increase MB:",round(rss1-rss0,2)); print("dtype:",next(pt_model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

Load s: 14.21 RSS increase MB: 37.74
dtype: torch.float32


In [ ]:
# CELL 10 — PyTorch CPU generation with KV cache
def generate_pytorch_cpu(text, max_length=MAX_OUTPUT_LENGTH):
    enc=build_encoder_text(text)
    t0=time.perf_counter(); inputs=tokenizer(enc,return_tensors="pt",max_length=MAX_INPUT_LENGTH,truncation=True); tok_s=time.perf_counter()-t0
    input_tokens=int(inputs["input_ids"].shape[-1]); t0=time.perf_counter()
    with torch.inference_mode():
        ids=pt_model.generate(**inputs,max_length=max_length,do_sample=False,num_beams=1,no_repeat_ngram_size=3,use_cache=True)
    infer_s=time.perf_counter()-t0; t0=time.perf_counter()
    raw=tokenizer.decode(ids[0],skip_special_tokens=False)
    if tokenizer.pad_token: raw=raw.replace(tokenizer.pad_token,"")
    if tokenizer.eos_token: raw=raw.replace(tokenizer.eos_token,"")
    raw=raw.strip(); output=postprocess_generated_text(raw); post_s=time.perf_counter()-t0
    out_tokens=int((ids[0]!=tokenizer.pad_token_id).sum().item())
    return {"output":output,"raw_output":raw,"input_tokens":input_tokens,"output_tokens":out_tokens,"latency_seconds":infer_s,"tokenization_seconds":tok_s,"postprocess_seconds":post_s,"tokens_per_second":out_tokens/infer_s if infer_s else np.nan}

In [ ]:
# CELL 11 — PyTorch sanity test + warmup
test_text="""Acme reported quarterly revenue of $4.2 billion, up 12% year over year. Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%. The company added 1.3 million customers and raised full-year revenue guidance from $16 billion to $17.5 billion. Management warned that European demand weakened in July."""
_=generate_pytorch_cpu(test_text); r=generate_pytorch_cpu(test_text)
print(r["output"]); print("Latency:",round(r["latency_seconds"],3),"s | tokens/s:",round(r["tokens_per_second"],2))

- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.
Latency: 2.015 s | tokens/s: 37.72


In [ ]:
# CELL 12 — Tune PyTorch intra-op CPU threads
physical=psutil.cpu_count(logical=False) or 1
candidates=[x for x in THREAD_CANDIDATES if x<=physical] or [1]
sample=eval_df.sample(n=min(10,len(eval_df)),random_state=SEED).reset_index(drop=True); rows=[]
for n in candidates:
    torch.set_num_threads(n); _=generate_pytorch_cpu(sample.iloc[0]["text"]); l=[]
    for _,row in sample.iterrows(): l.append(generate_pytorch_cpu(row["text"])["latency_seconds"])
    rows.append({"threads":n,"mean_latency_seconds":np.mean(l),"median_latency_seconds":np.median(l)})
thread_df=pd.DataFrame(rows).sort_values("median_latency_seconds"); display(thread_df)
BEST_THREADS=int(thread_df.iloc[0]["threads"]); torch.set_num_threads(BEST_THREADS); print("Selected:",BEST_THREADS)

,threads,mean_latency_seconds,median_latency_seconds
0,1,2.697127,1.84314


Selected: 1


In [ ]:
# CELL 13 — Shared generation benchmark loop
def benchmark_generator(fn, dataframe, name):
    rows=[]; _=fn(dataframe.iloc[0]["text"])
    for _,row in tqdm(dataframe.iterrows(),total=len(dataframe),desc=name):
        try:
            r=fn(row["text"]); rows.append({"example_id":row["example_id"],"source":row["source"],"text":row["text"],"reference":row["bullet_points"],"prediction":r["output"],"raw_output":r.get("raw_output",""),"input_tokens":r.get("input_tokens"),"output_tokens":r.get("output_tokens"),"latency_seconds":r.get("latency_seconds"),"tokenization_seconds":r.get("tokenization_seconds"),"postprocess_seconds":r.get("postprocess_seconds"),"tokens_per_second":r.get("tokens_per_second")})
        except Exception as e: rows.append({"example_id":row["example_id"],"source":row["source"],"text":row["text"],"reference":row["bullet_points"],"prediction":"","error":str(e)})
    return pd.DataFrame(rows)

In [ ]:
# CELL 14 — Benchmark PyTorch FP32 on all evaluation rows
pt_results=benchmark_generator(generate_pytorch_cpu,eval_df,"PyTorch-FP32-CPU")
pt_results.to_csv(RESULTS_DIR/"pytorch_fp32_raw.csv",index=False)
print("Mean latency:",pt_results.latency_seconds.mean(),"Median:",pt_results.latency_seconds.median(),"Mean tok/s:",pt_results.tokens_per_second.mean())

PyTorch-FP32-CPU:   0%|          | 0/500 [00:00<?, ?it/s]

Mean latency: 2.374760679334001 Median: 1.9513618194999935 Mean tok/s: 40.355791548033956


## Candidate B — ONNX Runtime FP32

ONNX Runtime changes the execution engine while preserving floating-point weights. This isolates runtime improvements from quantization improvements. For T5 generation, the exported model uses encoder/decoder components and a decoder-with-past path for cached decoding.

In [ ]:
# ============================================================
# CELL 1C — INSTALL ONNX + OPTIMUM ONNX
# ============================================================

!pip install -q \
    onnx \
    onnxruntime \
    psutil \
    optimum-onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 616.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# ============================================================
# CELL 15 — EXPORT T5 TO ONNX WITHOUT OPTIMUM PYTHON IMPORT
# ============================================================
#
# WHY THIS VERSION?
# -----------------
#
# Current error:
#
#   optimum.onnxruntime
#       ↓
#   tries importing AutoModelForVision2Seq
#       ↓
#   current Transformers does not expose it
#       ↓
#   ImportError
#
# This is a package-version compatibility issue, NOT a problem
# with our fine-tuned T5 model.
#
#
# We therefore avoid:
#
#   from optimum.onnxruntime import ORTModelForSeq2SeqLM
#
# in this cell.
#
# Instead we use the ONNX exporter through a subprocess.
#
# ============================================================

import os
import sys
import time
import shutil
import subprocess
from pathlib import Path


# ------------------------------------------------------------
# CLEAN EXPORT DIRECTORY
# ------------------------------------------------------------

ONNX_FP32_DIR = Path(
    "/content/t5_cpu_benchmark/onnx_fp32"
)


if ONNX_FP32_DIR.exists():

    shutil.rmtree(
        ONNX_FP32_DIR
    )


ONNX_FP32_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------

start = time.perf_counter()


command = [

    sys.executable,

    "-m",

    "transformers.onnx",

    "--model",
    MODEL_ID,

    "--feature",
    "seq2seq-lm-with-past",

    str(
        ONNX_FP32_DIR
    )
]


print(
    "Running ONNX export..."
)

print(
    " ".join(command)
)


result = subprocess.run(

    command,

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True
)


print(
    result.stdout
)


if result.returncode != 0:

    raise RuntimeError(
        "ONNX export failed. See output above."
    )


onnx_export_seconds = (
    time.perf_counter()
    -
    start
)


# ------------------------------------------------------------
# SAVE OUR EXACT TOKENIZER BESIDE ONNX MODEL
# ------------------------------------------------------------

tokenizer.save_pretrained(
    ONNX_FP32_DIR
)


# ------------------------------------------------------------
# SHOW GENERATED FILES
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    "ONNX EXPORT COMPLETE"
)

print("=" * 70)


onnx_files = list(
    ONNX_FP32_DIR.rglob(
        "*.onnx"
    )
)


for file in onnx_files:

    print(
        file.name,
        "->",
        round(
            file.stat().st_size
            /
            1024**2,
            2
        ),
        "MB"
    )


print()


print(
    "Export time:",
    round(
        onnx_export_seconds,
        2
    ),
    "seconds"
)


print(
    "Total directory size:",
    round(
        directory_size_mb(
            ONNX_FP32_DIR
        ),
        2
    ),
    "MB"
)


print(
    "ONNX files:",
    len(
        onnx_files
    )
)


assert len(
    onnx_files
) > 0, (
    "No ONNX files were created."
)

Running ONNX export...
/usr/bin/python3 -m transformers.onnx --model JayShah07/falconai-text-bullet-t5 --feature seq2seq-lm-with-past /content/t5_cpu_benchmark/onnx_fp32
2026-09-10 16:38:08.432132: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-10 16:38:10.199016: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Multiple distributions found for package optimum. Picked distribution: optimum-onnx
optimum.exporters.tasks.get_diffusers_tasks_to_model_mapping method failed to import diffusers with the error below.Please make sure you have diffusers installed and compatible with your transformers version.

Failed to import diffusers.pipelines.anyflow.pipeline_anyflow_far because of

AssertionError: No ONNX files were created.

In [ ]:
# CELL 16 — ONNX FP32 generation
def generate_onnx_fp32(text,max_length=MAX_OUTPUT_LENGTH):
    enc=build_encoder_text(text); t0=time.perf_counter(); inputs=tokenizer(enc,return_tensors="pt",max_length=MAX_INPUT_LENGTH,truncation=True); tok_s=time.perf_counter()-t0
    input_tokens=int(inputs["input_ids"].shape[-1]); t0=time.perf_counter()
    ids=ort_fp32_model.generate(**inputs,max_length=max_length,do_sample=False,num_beams=1,no_repeat_ngram_size=3,use_cache=True)
    infer_s=time.perf_counter()-t0; t0=time.perf_counter(); raw=tokenizer.decode(ids[0],skip_special_tokens=False)
    if tokenizer.pad_token: raw=raw.replace(tokenizer.pad_token,"")
    if tokenizer.eos_token: raw=raw.replace(tokenizer.eos_token,"")
    raw=raw.strip(); output=postprocess_generated_text(raw); post_s=time.perf_counter()-t0
    out_tokens=int((ids[0]!=tokenizer.pad_token_id).sum().item())
    return {"output":output,"raw_output":raw,"input_tokens":input_tokens,"output_tokens":out_tokens,"latency_seconds":infer_s,"tokenization_seconds":tok_s,"postprocess_seconds":post_s,"tokens_per_second":out_tokens/infer_s if infer_s else np.nan}

In [ ]:
# CELL 17 — ONNX FP32 sanity + full benchmark
_=generate_onnx_fp32(test_text); r=generate_onnx_fp32(test_text); print(r["output"]); print("Latency:",round(r["latency_seconds"],3))
ort_fp32_results=benchmark_generator(generate_onnx_fp32,eval_df,"ONNX-FP32-CPU"); ort_fp32_results.to_csv(RESULTS_DIR/"onnx_fp32_raw.csv",index=False)

## Candidate C — ONNX Runtime dynamic INT8

Dynamic INT8 is the first deployment quantization candidate: it reduces weight storage substantially and often improves CPU matrix-multiplication efficiency without retraining. We quantize each exported ONNX graph and then repeat the identical benchmark.

In [ ]:
# CELL 18 — Dynamic INT8 quantization
from onnxruntime.quantization import quantize_dynamic, QuantType
shutil.rmtree(ONNX_INT8_DIR,ignore_errors=True); shutil.copytree(ONNX_FP32_DIR,ONNX_INT8_DIR)
onnx_files=sorted(ONNX_INT8_DIR.glob("*.onnx")); print([x.name for x in onnx_files]); assert onnx_files
for src in onnx_files:
    tmp=src.with_name(src.stem+"_int8_tmp.onnx"); print("Quantizing",src.name)
    quantize_dynamic(str(src),str(tmp),weight_type=QuantType.QInt8,per_channel=True,reduce_range=False)
    src.unlink(); tmp.rename(src)
print("INT8 size MB:",round(directory_size_mb(ONNX_INT8_DIR),2))

In [ ]:
# CELL 19 — Load INT8 ONNX model
rss0=process_rss_mb(); start=time.perf_counter()
ort_int8_model=ORTModelForSeq2SeqLM.from_pretrained(ONNX_INT8_DIR,provider="CPUExecutionProvider",use_cache=True)
int8_load_seconds=time.perf_counter()-start; rss1=process_rss_mb()
print("Load s:",round(int8_load_seconds,3),"RSS increase MB:",round(rss1-rss0,2))

In [ ]:
# CELL 20 — INT8 generation
def generate_onnx_int8(text,max_length=MAX_OUTPUT_LENGTH):
    enc=build_encoder_text(text); t0=time.perf_counter(); inputs=tokenizer(enc,return_tensors="pt",max_length=MAX_INPUT_LENGTH,truncation=True); tok_s=time.perf_counter()-t0
    input_tokens=int(inputs["input_ids"].shape[-1]); t0=time.perf_counter()
    ids=ort_int8_model.generate(**inputs,max_length=max_length,do_sample=False,num_beams=1,no_repeat_ngram_size=3,use_cache=True)
    infer_s=time.perf_counter()-t0; t0=time.perf_counter(); raw=tokenizer.decode(ids[0],skip_special_tokens=False)
    if tokenizer.pad_token: raw=raw.replace(tokenizer.pad_token,"")
    if tokenizer.eos_token: raw=raw.replace(tokenizer.eos_token,"")
    raw=raw.strip(); output=postprocess_generated_text(raw); post_s=time.perf_counter()-t0
    out_tokens=int((ids[0]!=tokenizer.pad_token_id).sum().item())
    return {"output":output,"raw_output":raw,"input_tokens":input_tokens,"output_tokens":out_tokens,"latency_seconds":infer_s,"tokenization_seconds":tok_s,"postprocess_seconds":post_s,"tokens_per_second":out_tokens/infer_s if infer_s else np.nan}

In [ ]:
# CELL 21 — INT8 sanity + full benchmark
_=generate_onnx_int8(test_text); r=generate_onnx_int8(test_text); print(r["output"]); print("Latency:",round(r["latency_seconds"],3))
ort_int8_results=benchmark_generator(generate_onnx_int8,eval_df,"ONNX-INT8-CPU"); ort_int8_results.to_csv(RESULTS_DIR/"onnx_int8_raw.csv",index=False)

## Quality scoring

BERTScore is intentionally outside the timed inference loop because it is an evaluator, not part of production inference. On CPU it may take noticeable time.

In [ ]:
# CELL 22 — BERTScore evaluator on CPU
bert_scorer=BERTScorer(model_type="distilbert-base-uncased",lang="en",device="cpu",rescale_with_baseline=False)
print("BERTScore ready")

In [ ]:
# CELL 23 — Common quality scorer
def score_results(df,name):
    s=df.copy(); r1=[];r2=[];rl=[]
    for _,row in tqdm(s.iterrows(),total=len(s),desc="ROUGE - "+name):
        x=rouge.score(row.reference,row.prediction); r1.append(x["rouge1"].fmeasure);r2.append(x["rouge2"].fmeasure);rl.append(x["rougeL"].fmeasure)
    s["rouge1"]=r1;s["rouge2"]=r2;s["rougeL"]=rl
    P,R,F=bert_scorer.score(s.prediction.tolist(),s.reference.tolist()); s["bertscore_precision"]=P.numpy();s["bertscore_recall"]=R.numpy();s["bertscore_f1"]=F.numpy()
    s["bullet_format"]=s.prediction.apply(bullet_format_score);s["predicted_bullets"]=s.prediction.apply(count_bullets);s["reference_bullets"]=s.reference.apply(count_bullets);s["bullet_count_error"]=(s.predicted_bullets-s.reference_bullets).abs()
    s["input_words"]=s.text.apply(word_count);s["prediction_words"]=s.prediction.apply(word_count);s["reference_words"]=s.reference.apply(word_count); safe=s.input_words.clip(lower=1);s["compression_ratio"]=s.prediction_words/safe;s["reference_compression_ratio"]=s.reference_words/safe
    lat=s.latency_seconds.dropna();tps=s.tokens_per_second.dropna()
    summary={"model":name,"examples":len(s),"rouge1":s.rouge1.mean(),"rouge2":s.rouge2.mean(),"rougeL":s.rougeL.mean(),"bertscore_precision":s.bertscore_precision.mean(),"bertscore_recall":s.bertscore_recall.mean(),"bertscore_f1":s.bertscore_f1.mean(),"bullet_format":s.bullet_format.mean(),"avg_predicted_bullets":s.predicted_bullets.mean(),"avg_reference_bullets":s.reference_bullets.mean(),"mean_bullet_count_error":s.bullet_count_error.mean(),"compression_ratio":s.compression_ratio.mean(),"reference_compression_ratio":s.reference_compression_ratio.mean(),"avg_latency_seconds":lat.mean(),"median_latency_seconds":lat.median(),"p95_latency_seconds":lat.quantile(.95),"avg_output_tokens":s.output_tokens.mean(),"avg_tokens_per_second":tps.mean(),"median_tokens_per_second":tps.median()}
    return s,summary

In [ ]:
# CELL 24 — Score all candidates
pt_scored,pt_summary=score_results(pt_results,"PyTorch-FP32-CPU")
ort_fp32_scored,ort_fp32_summary=score_results(ort_fp32_results,"ONNX-FP32-CPU")
ort_int8_scored,ort_int8_summary=score_results(ort_int8_results,"ONNX-INT8-CPU")
pt_scored.to_csv(RESULTS_DIR/"pytorch_fp32_scored.csv",index=False);ort_fp32_scored.to_csv(RESULTS_DIR/"onnx_fp32_scored.csv",index=False);ort_int8_scored.to_csv(RESULTS_DIR/"onnx_int8_scored.csv",index=False)

In [ ]:
# CELL 25 — Add artifact sizes and load times
PT_LOCAL_DIR=BASE_DIR/"pytorch_fp32_local"
if not PT_LOCAL_DIR.exists(): pt_model.save_pretrained(PT_LOCAL_DIR); tokenizer.save_pretrained(PT_LOCAL_DIR)
pt_size=directory_size_mb(PT_LOCAL_DIR); fp32_size=directory_size_mb(ONNX_FP32_DIR); int8_size=directory_size_mb(ONNX_INT8_DIR)
pt_summary.update({"artifact_size_mb":pt_size,"model_load_seconds":pt_load_seconds});ort_fp32_summary.update({"artifact_size_mb":fp32_size,"model_load_seconds":onnx_export_seconds});ort_int8_summary.update({"artifact_size_mb":int8_size,"model_load_seconds":int8_load_seconds})
print("PyTorch FP32:",round(pt_size,2),"MB | ONNX FP32:",round(fp32_size,2),"MB | ONNX INT8:",round(int8_size,2),"MB")

In [ ]:
# CELL 26 — Final quality/performance report
report_df=pd.DataFrame([pt_summary,ort_fp32_summary,ort_int8_summary])
cols=["model","artifact_size_mb","model_load_seconds","rouge1","rouge2","rougeL","bertscore_precision","bertscore_recall","bertscore_f1","bullet_format","avg_predicted_bullets","avg_reference_bullets","mean_bullet_count_error","compression_ratio","reference_compression_ratio","avg_latency_seconds","median_latency_seconds","p95_latency_seconds","avg_output_tokens","avg_tokens_per_second"]
report_df=report_df[cols]; display(report_df)

In [ ]:
# CELL 27 — INT8 quality delta and acceptance gate
base=pt_summary;cand=ort_int8_summary; metrics=["rouge1","rouge2","rougeL","bertscore_f1"]; rows=[]
for m in metrics:
    b=float(base[m]);c=float(cand[m]);rows.append({"metric":m,"fp32":b,"int8":c,"absolute_delta":c-b,"relative_degradation_pct":((b-c)/b*100) if b else np.nan})
delta_df=pd.DataFrame(rows);display(delta_df);d=dict(zip(delta_df.metric,delta_df.relative_degradation_pct))
quality_pass=d["rougeL"]<=2.0 and d["rouge2"]<=2.0 and d["bertscore_f1"]<=1.0
print("INT8 QUALITY GATE:","PASS" if quality_pass else "FAIL")

In [ ]:
# CELL 28 — Performance and size improvement
perf_report=pd.DataFrame([
{"comparison":"ONNX FP32 vs PyTorch FP32","latency_speedup_x":pt_summary["median_latency_seconds"]/ort_fp32_summary["median_latency_seconds"],"size_reduction_pct":100*(1-fp32_size/pt_size)},
{"comparison":"ONNX INT8 vs PyTorch FP32","latency_speedup_x":pt_summary["median_latency_seconds"]/ort_int8_summary["median_latency_seconds"],"size_reduction_pct":100*(1-int8_size/pt_size)},
{"comparison":"ONNX INT8 vs ONNX FP32","latency_speedup_x":ort_fp32_summary["median_latency_seconds"]/ort_int8_summary["median_latency_seconds"],"size_reduction_pct":100*(1-int8_size/fp32_size)}]);display(perf_report)

In [ ]:
# CELL 29 — Check how often quantization changes exact text
cmp=pd.DataFrame({"example_id":pt_results.example_id,"fp32_prediction":pt_results.prediction,"int8_prediction":ort_int8_results.prediction});cmp["exact_match"]=cmp.fp32_prediction==cmp.int8_prediction
print("Exact same output %:",round(cmp.exact_match.mean()*100,2));print("Changed:",(~cmp.exact_match).sum());display(cmp[~cmp.exact_match].head(10))

In [ ]:
# CELL 30 — Per-source INT8 report
source_report=ort_int8_scored.groupby("source").agg(examples=("example_id","count"),rouge1=("rouge1","mean"),rouge2=("rouge2","mean"),rougeL=("rougeL","mean"),bertscore_f1=("bertscore_f1","mean"),latency_seconds=("latency_seconds","mean"),tokens_per_second=("tokens_per_second","mean")).reset_index().sort_values("rougeL",ascending=False)
display(source_report)

In [ ]:
# CELL 31 — Save all final reports
report_df.to_csv(RESULTS_DIR/"cpu_quantization_report.csv",index=False);delta_df.to_csv(RESULTS_DIR/"int8_quality_delta.csv",index=False);perf_report.to_csv(RESULTS_DIR/"performance_improvement.csv",index=False);source_report.to_csv(RESULTS_DIR/"int8_per_source_report.csv",index=False);cmp.to_csv(RESULTS_DIR/"fp32_vs_int8_predictions.csv",index=False)
print("Saved reports to",RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*")): print(" -",p.name)